<a href="https://colab.research.google.com/github/Urooj25/Movie-Recommender-System/blob/main/Copy_of_Movies_recommendation_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
movies = pd.read_csv('/content/tmdb_5000_movies.csv')
movies.head()
#genere,id,keywords,title,overview,
movies = movies[["id","title","overview","genres","keywords"]]
movies.head()
#preprocessing
movies.isnull().sum()
movies.dropna(inplace=True)
movies.duplicated().sum()
import ast
#helper function
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):#convert dict into list
        L.append(i['name'])
    return L

#convert into list
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

movies.head()
movies['overview'][0]
movies['overview'].apply(lambda x:x.split())
movies.head()
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['overview'] = movies['overview'].apply(lambda x:x.split())
movies.head()
movies["tags"] = movies['overview'] + movies["genres"] + movies["keywords"]
movies.head()
new_df = movies[['id','title','tags']]
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))
new_df.head()
new_df['tags'][0]
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())
new_df.head()
#text vectorization
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()
print(vectors.shape)
print(vectors[0])
feature_names = cv.get_feature_names_out()
print(len(feature_names))
#steming same word
# 1. Stemming
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

new_df['tags'] = new_df['tags'].apply(stem)

# 2. Vectorization
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

# 3. Cosine Similarity
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)

# 4. Recommend Function
def recommend(movie):
    try:
        movie_index = new_df[new_df['title'] == movie].index[0]
        distances = similarity[movie_index]
        movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

        print(f"Recommendations for '{movie}':")
        for i in movies_list:
            print("-", new_df.iloc[i[0]].title)
    except IndexError:
        print("Movie not found in dataset. Please check spelling!")

# Test
recommend('The Princess and the Frog')


In [ ]:
from google.colab import files
files.download('similarity.pkl')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>